In [1]:
import os
from dotenv import load_dotenv
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types
from typing import Optional,Dict,Any

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.CRITICAL)

print("Libraries imported")

Libraries imported


In [2]:
# 1. 加载同目录下的 .env 文件
load_dotenv()

# 2. 从系统环境变量中读取模型名称（如果没读到，默认用 deepseek-chat）
# 注意：LiteLlm 在底层会自动去寻找 os.environ["DEEPSEEK_API_KEY"]，所以我们甚至不需要手动赋给它！
MODEL_NAME = os.getenv("DEEPSEEK_MODEL", "deepseek/deepseek-chat")
llm = LiteLlm(model=MODEL_NAME)

print(llm.llm_client.completion(model=llm.model,
                                messages=[{"role": "user", "content": "你好，请问你准备好开始了吗？"}],
                                tools=[]))
print("🤖 DeepSeek 回复：")

print("\nDeepSeek is ready for use.")

ModelResponse(id='0f07d4c0-2833-4621-9cbd-90b696bfdc2c', created=1788081276, model='deepseek-v4-flash', object='chat.completion', system_fingerprint='a26a7955944dc5c60445bff77fac9c8e', choices=[Choices(finish_reason='stop', index=0, message=Message(content='你好！我已经准备好了，随时可以开始。无论是回答问题、提供建议，还是帮你处理某些任务，我都会尽力协助。请告诉我你现在需要什么帮助吧！😊', role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None), provider_specific_fields={})], usage=Usage(completion_tokens=35, prompt_tokens=12, total_tokens=47, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None), prompt_cache_hit_tokens=0, prompt_cache_miss_tokens=12))
🤖 DeepSeek 回复：

DeepSeek is ready for use.


In [3]:
# 1. 导入刚才 pip install 安装好的官方库
from neo4j_for_adk import graphdb

neo4j_is_ready = graphdb.send_query("RETURN 'Neo4j is Ready' as message")

print("连接测试结果：", neo4j_is_ready)

连接测试结果： {'status': 'success', 'query_result': [{'message': 'Neo4j is Ready'}]}


In [14]:
##定义Agent工具

In [4]:
def hello_world_tool(name: str) -> dict:
    """
    这是一个打招呼的工具。当用户要求你向某人说 Hello 或者打招呼时，请调用此工具。

    Args:
        name (str): 需要打招呼的人的名字。

    Returns:
        dict: 包含打招呼结果的字典。
    """
    # 这里是工具的核心逻辑（你可以把它想象成查数据库、调 API 的地方）
    greeting_message = f"Hello, {name}! 欢迎来到 Google ADK 的世界！"

    # 打印一条信息在控制台，方便我们肉眼观察工具是否真的被偷偷调用了
    print(f"\n[🔧 后台运行] hello_world_tool 被触发了，参数 name={name}")

    cypher_query = f"RETURN 'Hello to you, {name}' AS reply"

    return graphdb.send_query(cypher_query)

In [5]:
print(hello_world_tool("ADK"))


[🔧 后台运行] hello_world_tool 被触发了，参数 name=ADK
{'status': 'success', 'query_result': [{'reply': 'Hello to you, ADK'}]}


In [17]:
##定义Agent

In [68]:
from pathlib import Path

# 加载指令文件
PROMPT_FILE = Path(__file__).parent / "greeting_subagent_instruction.md"
GREETING_SUBAGENT_INSTRUCTION = PROMPT_FILE.read_text(encoding="utf-8")

greeting_agent = Agent(
    name="GreetingAgent",  # Agent 的名字

    # 【新增】：Agent 的“个人简历”或“名片”。
    # 作用：以后如果有另一个调度 Agent 需要找人打招呼，看到这段描述就会把任务派给它。
    description="这是一个专职的迎宾专员智能体。它的主要职责是使用专用的工具向新用户发送问候和打招呼。当有欢迎新客人的需求时，请呼叫此 Agent。",

    # Agent 自己的“工作手册”。告诉它拿到任务后具体该怎么做。
    instruction=GREETING_SUBAGENT_INSTRUCTION,
    model=llm,
    tools=[hello_world_tool]
)

print(f"Agent '{greeting_agent.name}' created")


Agent 'GreetingAgent' created


In [19]:
##创建Runner和SessionService

In [69]:
app_name = greeting_agent.name + "_app"
user_id = greeting_agent.name + "_user"
session_id = greeting_agent.name + "_session_01"


# ================= 4. 测试你的 Agent =================
print("================ 测试开始 ================\n")

user_input = "嘿，今天有位新朋友叫 '周泊旭' 来了，请你代表我们向他打个招呼吧！"
print(f"👤 用户: {user_input}")

# 1. 实例化一个“内存会话服务”
session_service = InMemorySessionService()
await session_service.create_session(
    app_name=app_name,
    user_id=user_id,
    session_id=session_id
)

# 2. 实例化 Runner（核心修改：允许自动创建会话！）
runner = Runner(
    app_name=app_name,
    agent=greeting_agent,
    session_service=session_service,
)

# 3. 将普通文本包装成标准的 UserContent
message = types.UserContent(parts=[types.Part(text=user_input)])

final_response_text = "智能体没有生成最终响应"

print("\n🤖 Agent 正在思考并执行中...\n")

# 4. 运行 Agent
verbose = False
async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=message):
    if verbose:
        print(f"  [Event] Author: {event.author}, Type: {type(event).__name__}, Final: {event.is_final_response()}, Content: {event.content}")

    # # 提取 Agent 说的话并打印出来
    if getattr(event, "is_final_response", False):
        if getattr(event, "content", None) and event.content.parts:
            final_response_text = event.content.parts[0].text
        elif getattr(event, "actions", None) and getattr(event.actions, "escalate", None):
            final_response_text = f"Agent escalated: {getattr(event, 'error_message', 'No specific message')}"

# 打印内容，并且不换行，实现打字机效果
print(f"<<< Agent Response: {final_response_text}")

print("\n\n================ 测试结束 ================")

================ 测试开始 ================

👤 用户: 嘿，今天有位新朋友叫 '周泊旭' 来了，请你代表我们向他打个招呼吧！

🤖 Agent 正在思考并执行中...



KeyError: 'Context variable not found: `user_real_name`.'

In [ ]:
# 创建Agent Caller类

In [70]:
class AgentCaller:
    def __init__(self, agent: Agent, runner: Runner, user_id: str, session_id: str, verbose: bool = False):
        """
        初始化 Agent 调用器
        """
        self.verbose = verbose
        self.agent = agent
        # 如果不传参数，就自动根据 agent 的名字生成默认 ID
        self.app_name = app_name
        self.user_id = user_id
        self.session_id = session_id
        self.runner = runner

        # 实例化基础服务
        self.session_service = InMemorySessionService()

        # 内部状态标志，用于确保只创建一次 Session
        self._session_created = False

    async def chat(self, user_input: str) -> str:
        """
        发送消息并获取 Agent 的回复
        """
        # 1. 确保在第一次聊天时创建好 Session
        if not self._session_created:
            await self.session_service.create_session(
                app_name=self.app_name,
                user_id=self.user_id,
                session_id=self.session_id
            )
            self._session_created = True

        print(f"\n>>>👤 用户: {user_input}")
        if self.verbose:
            print("\n🤖 Agent 正在思考并执行中...\n")

        # 2. 组装输入消息
        message = types.Content(role='user', parts=[types.Part(text=user_input)])
        final_response_text = "智能体没有生成最终响应"

        # 3. 异步运行 Agent，抓取事件流
        async for event in self.runner.run_async(user_id=self.user_id, session_id=self.session_id, new_message=message):

            # 调试模式打印所有事件详情
            if self.verbose:
                is_final = getattr(event, "is_final_response", False)
                print(f"  [Event] Author: {getattr(event, 'author', 'N/A')}, Type: {type(event).__name__}, Final: {is_final}, Content: {getattr(event, 'content', 'N/A')}")

            # 4. 提取最终的回复内容
            if getattr(event, "is_final_response", False):
                if getattr(event, "content", None) and event.content.parts:
                    final_response_text = event.content.parts[0].text
                elif getattr(event, "actions", None) and getattr(event.actions, "escalate", None):
                    final_response_text = f"Agent escalated: {getattr(event, 'error_message', 'No specific message')}"

                # 抓到最终回复后，直接跳出循环
                break

        print(f"<<< Agent Response: {final_response_text}\n")
        return final_response_text

In [ ]:
# 创建AgentCaller实例

In [71]:
async def make_agent_caller(agent: Agent,                                  # 指定必须传一个 Agent 对象
    app_name: Optional[str] = None,
    initial_state: Optional[Dict[str, Any]] = None) -> AgentCaller:
    # 1. 搞定名字
    app_name = app_name or f"{agent.name}_app"
    user_id = f"{agent.name}_user"
    session_id = f"{agent.name}_session"

    # 2. 搞定异步的会话创建（把需要 await 的脏活累活全在这里做完）
    session_service = InMemorySessionService()
    await session_service.create_session(
        app_name=app_name,
        user_id=user_id,
        session_id=session_id,
        state=initial_state
    )

    # 3. 组装 Runner
    runner = Runner(
        app_name=app_name,
        agent=agent,
        session_service=session_service
    )

    # 4. 关键：把准备好的东西塞进对象里，并返回这个完美的对象！
    return AgentCaller(agent=agent, runner=runner, user_id=user_id, session_id=session_id)

In [ ]:
# 测试AgentCaller

In [72]:
# 测试：我们偷偷给 Agent 植入一个初始状态（背景信息）
my_state = {
    "current_user_vip_level": "超级金牌会员",
    "user_real_name": "周泊旭"
}

# 制造 AgentCaller，并把背景信息塞进去
caller = await make_agent_caller(
    agent=greeting_agent,
    initial_state=my_state
)

# 开始聊天（这时候就算你不提你的名字和等级，Agent 底层其实也已经拥有这些数据了！）
async def run_conversation():
    await caller.chat("你好！请问我是谁")

    await caller.chat("我的现在的vip等级是什么")

await run_conversation()


>>>👤 用户: 你好！请问我是谁
<<< Agent Response: 尊敬的周泊旭先生，您好！🌟

非常荣幸能在这里迎接您！让我确认一下您的身份信息——

根据系统显示，您的尊姓大名是 **周泊旭**，而且您是我们尊贵的 **超级金牌会员**！👑

能为您服务，真是让我感到无比荣幸！作为我们的超级金牌会员，您将享受到我们最顶级的专属服务与礼遇。请问今天有什么我可以为您效劳的吗？无论是需要帮助还是想要了解一下我们的服务，我都随时待命！✨


>>>👤 用户: 我的现在的vip等级是什么
<<< Agent Response: 让我为您确认一下您的VIP等级信息！

根据系统显示，您目前是 **超级金牌会员**！👑

作为我们最尊贵的超级金牌会员，您享受的可是最高级别的礼遇和专属服务哦～无论是特别通道、专属客服还是各种超值优惠，统统为您安排得妥妥的！✨

请问还有什么可以为您效劳的吗，周泊旭先生？我非常乐意为您服务！🌟

